In [16]:
from sentence_transformers import SentenceTransformer
from pydantic import BaseModel, Field
from typing import Any, Optional
import re
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd
from ddgs import DDGS
import os 
from dotenv import load_dotenv
from openai import OpenAI

In [3]:
class FAQEntry(BaseModel):
    id: int
    question: str
    answer: str
    text: str 

class SearchResult(BaseModel):
    entry_id: int
    question: str
    answer: str
    score: float = Field(ge=-1.0, le=1.0)

class Answer(BaseModel):
    answer: str
    sources: list[int]
    could_answer: bool

class Trace(BaseModel):
    tool_name: str
    args: dict[str, Any]
    result_ids: list[int] = Field(default_factory=list)
    result_scores: list[float] = Field(default_factory=list)
    elapsed_ms: int
    requested_by_model: bool = True 

In [4]:
faq_text = open('./data/devcolorfaq.txt').read()
entries = re.split(r'\n\n+', faq_text.strip())
print(entries[0])

1. **What is the mission of /dev/color?**  
   /dev/color is dedicated to supporting and empowering Black technologists by fostering a strong community, providing career development resources, and advocating for diversity in the tech industry. The organization aims to help Black software engineers, founders, and leaders navigate challenges and advance their careers. Through mentorship, programs, and industry collaborations, /dev/color strives to create lasting change in the tech sector.


In [5]:
entry = entries[0]
match = re.match(r'\d+\.\s*\*\*(.+?)\*\*\s*\n(.+)', entry, re.DOTALL)
print("Q:", match.group(1).strip())
print("A:", match.group(2).strip())

Q: What is the mission of /dev/color?
A: /dev/color is dedicated to supporting and empowering Black technologists by fostering a strong community, providing career development resources, and advocating for diversity in the tech industry. The organization aims to help Black software engineers, founders, and leaders navigate challenges and advance their careers. Through mentorship, programs, and industry collaborations, /dev/color strives to create lasting change in the tech sector.


In [6]:
result = []
for i, entry in enumerate(entries):
    match = re.match(r'\d+\.\s*\*\*(.+?)\*\*\s*\n(.+)', entry, re.DOTALL)
    q = match.group(1).strip()
    a = match.group(2).strip()
    result.append(FAQEntry(id=i+1, question=q, answer=a, text=f"Question: {q}\nAnswer: {a}"))

In [7]:
result[0].text

'Question: What is the mission of /dev/color?\nAnswer: /dev/color is dedicated to supporting and empowering Black technologists by fostering a strong community, providing career development resources, and advocating for diversity in the tech industry. The organization aims to help Black software engineers, founders, and leaders navigate challenges and advance their careers. Through mentorship, programs, and industry collaborations, /dev/color strives to create lasting change in the tech sector.'

In [8]:
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode([r.text for r in result])


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5258.58it/s]


In [9]:
embeddings.shape

(10, 384)

In [10]:
query = "How can /dev/color help me develop my career?"
query_embedding = model.encode(query)
scores = cosine_similarity(query_embedding.reshape(1, -1), embeddings)

In [13]:
import pandas as pd

df = pd.DataFrame(
    {
        "entry_id": [e.id for e in result],
        "question": [e.question for e in result],
        "answer": [e.answer for e in result],
        "cosine_similarity": scores[0].astype(float),
    }
).sort_values("cosine_similarity", ascending=False).reset_index(drop=True)
print(f"Query: {query}")
df

Query: How can /dev/color help me develop my career?


,entry_id,question,answer,cosine_similarity
0,1,What is the mission of /dev/color?,/dev/color is dedicated to supporting and empo...,0.735279
1,10,How can individuals or companies contribute to...,Individuals can support /dev/color by making f...,0.673891
2,6,How does /dev/color collaborate with other org...,"In 2023, /dev/color partnered with organizatio...",0.658789
3,5,What types of events and networking opportunit...,/dev/color hosted six in-person events across ...,0.643146
4,2,What key achievements did /dev/color members a...,"In 2023, /dev/color members celebrated numerou...",0.615914
5,8,What are some key statistics on member engagem...,"In 2023, /dev/color had 760 active members acr...",0.554559
6,7,What are the main sources of funding for /dev/...,/dev/color's primary funding comes from corpor...,0.550841
7,9,Which corporate partners supported /dev/color ...,Some of /dev/color’s key corporate partners in...,0.517246
8,3,"What is the A* Program, and how does it suppor...",The A* Program is /dev/color’s flagship initia...,0.496399
9,4,What impact has the Executive Accelerator Prog...,"Launched in 2023, the Executive Accelerator Pr...",0.293086


In [27]:
class FAQIndex:
    def __init__(self, faq_text: str):
        self.model = SentenceTransformer('all-MiniLM-L6-v2')
        self.entries = self._parse(faq_text)
        self.embeddings = self.model.encode([e.text for e in self.entries])

    def _parse(self, faq_text: str) -> list[FAQEntry]:
        raw = re.split(r'\n(?=\d+\.)', faq_text.strip())
        entries = []
        for i, entry in enumerate(raw):
            match = re.match(r'\d+\.\s*\*\*(.+?)\*\*\s*\n(.+)', entry, re.DOTALL)
            q = match.group(1).strip()
            a = match.group(2).strip()
            entries.append(FAQEntry(id=i+1, question=q, answer=a, text=f"Question: {q}\nAnswer: {a}"))
        return entries

    def search(self, query: str, top_k: int = 3) -> list[SearchResult]:
        query_emb = self.model.encode(query)
        scores = cosine_similarity([query_emb], self.embeddings)[0]
        top_idx = np.argsort(scores)[::-1][:top_k]
        return [
            SearchResult(
                entry_id=self.entries[i].id,
                question=self.entries[i].question,
                answer=self.entries[i].answer,
                score=float(scores[i])
            ) for i in top_idx
        ]


In [28]:
index = FAQIndex(open('./data/devcolorfaq.txt').read())
index.search("How can /dev/color help me develop my career?")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8771.13it/s]


[SearchResult(entry_id=1, question='What is the mission of /dev/color?', answer='/dev/color is dedicated to supporting and empowering Black technologists by fostering a strong community, providing career development resources, and advocating for diversity in the tech industry. The organization aims to help Black software engineers, founders, and leaders navigate challenges and advance their careers. Through mentorship, programs, and industry collaborations, /dev/color strives to create lasting change in the tech sector.', score=0.7352786064147949),
 SearchResult(entry_id=10, question='How can individuals or companies contribute to /dev/color’s mission?', answer='Individuals can support /dev/color by making financial contributions, participating in mentorship programs, and advocating for diversity in tech. Companies can collaborate by sponsoring events, providing funding, or offering professional development opportunities for Black technologists. These contributions help sustain the org

In [15]:
def web_search(query: str) -> list[dict]:
    return DDGS().text(query, max_results=3)
web_search("How can /dev/color help me develop my career?")

[{'title': "/dev/color: Becoming A*. One software engineer's ...",
  'href': 'https://medium.com/@aculver28/dev-color-becoming-a-9ab50a38b5d8',
  'body': 'August 30, 2020 - Then one day, in the midst of a hardcore procrastination session at work, I stumbled across the A* program facilitated by an organization named /dev/color. ... An organization focused on connecting Black software engineers in an effort to support and help each other achieve their professional goals? Like, in real life??? I’m absolutely here for that. Fast-forward through the information sessions, applications, etc. and I’m walking into a room to begin my journey as a member of the inaugural cohort of the Atlanta chapter of the A* program.'},
 {'title': 'A* - DevColor',
  'href': 'https://devcolor.org/a-program/',
  'body': 'December 15, 2024 - Simply put, this program is for people who ship code or manage others who do. A* is available exclusively to professional and entrepreneur members of /dev/color who are curren

In [20]:
load_dotenv()
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"]
)

In [21]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_faq",
            "description": "Search the /dev/color FAQ knowledge base",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The search query"},
                    "top_k": {"type": "integer", "description": "Number of results", "default": 3}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Search the web for information not found in the FAQ",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The web search query"}
                },
                "required": ["query"]
            }
        }
    }
]


In [22]:
messages = [
    {"role": "system", "content": "You are a helpful assistant for /dev/color. Use the search_faq tool for questions about /dev/color, and web_search for other questions."},
    {"role": "user", "content": "How can /dev/color help me develop my career?"}
]

response = client.chat.completions.create(
    model="openai/gpt-4o-mini",
    messages=messages,
    tools=tools
)

print(response.choices[0].message)

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_nloSWPapECkf1hiMkIgk0e9t', function=Function(arguments='{"query":"career development"}', name='search_faq'), type='function', index=0)], reasoning=None)


In [24]:
import json
tc = response.choices[0].message.tool_calls[0]
print(tc.function.name)
print(json.loads(tc.function.arguments))

search_faq
{'query': 'career development'}


In [29]:
if tc.function.name == "search_faq":
    result = index.search(**json.loads(tc.function.arguments))
    print(result)


[SearchResult(entry_id=4, question='What impact has the Executive Accelerator Program had on Black leaders in tech?', answer='Launched in 2023, the Executive Accelerator Program supports rising Black executives by providing mentorship, coaching, and executive training. Participants engaged in a mix of virtual and in-person sessions, gaining skills in leadership, networking, and board service preparation. As a result, 100% of participants reported making progress toward their career goals, with many feeling more confident and empowered in their leadership roles.', score=0.31382685899734497), SearchResult(entry_id=2, question='What key achievements did /dev/color members accomplish in 2023?', answer='In 2023, /dev/color members celebrated numerous personal and professional milestones, including promotions, new job opportunities, and high-profile appearances. Many members reported feeling a stronger sense of community, professional support, and increased access to opportunities. The organ

In [30]:
SYSTEM_PROMPT = """You are a helpful assistant for /dev/color. 
                   Use the search_faq tool for questions about /dev/color. 
                   If the FAQ doesn't have a good answer, ask the user if they'd like you to search the web before using web_search."""

In [31]:
def run_agent(query: str, verbose: bool = False):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": query}
    ]
    while True:
        response = client.chat.completions.create(
            model="openai/gpt-4o-mini",
            messages=messages,
            tools=tools
        )
        msg = response.choices[0].message 
        if not msg.tool_calls:
            return msg.content 
        messages.append(msg)
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            if verbose:
                print(f"🔧 Calling {tc.function.name} ({args})")
            
            if tc.function.name == "search_faq":
                result = index.search(**args)
                tool_result = json.dumps([r.model_dump() for r in result if r.score > 0.5])
            elif tc.function.name == "web_search":
                result = web_search(**args) 
                tool_result = json.dumps(result)
            if verbose:
                print(f"📄 Got {len(result)} results")
            
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": tool_result
            })

In [32]:
run_agent("How can /dev/color help me develop my career?", verbose=True)

🔧 Calling search_faq ({'query': '/dev/color career development'})
📄 Got 3 results


'/dev/color can help you develop your career in several ways:\n\n1. **Community Support**: It provides a strong community for Black technologists to connect, share experiences, and support each other.\n\n2. **Career Development Resources**: The organization offers various resources aimed at helping Black software engineers, founders, and leaders navigate their careers and advance in the tech industry.\n\n3. **Mentorship**: Through mentorship programs, you can receive guidance from experienced professionals who can help you navigate challenges and make informed career decisions.\n\n4. **Networking Opportunities**: /dev/color hosts events and networking opportunities which allow you to connect with peers, industry leaders, and corporate partners. These events create spaces to learn and grow in a supportive environment.\n\n5. **Collaborations with Industry Leaders**: The organization collaborates with tech companies and other organizations to provide panels, professional development sessi